In [ ]:
import flwr as fl
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.optimizers.legacy import Adam
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay



In [ ]:

def load_data(filename):
    global X, y, df
    df = pd.read_csv(filename)
    X = df.iloc[:, :-1].values  # Features: all columns except the last one
    y = df.iloc[:, -1].values  # Labels: the last column

    return X, y 


In [ ]:
def build_dnn_model(output_units=5):  # Adjust `output_units` based on your specific task
    model = tf.keras.Sequential([
        # Input layer with increased units and input shape for 24 features
        tf.keras.layers.Dense(512, activation='relu', input_shape=(23,)),
        tf.keras.layers.Dropout(0.2),  # Increased dropout for regularization
        
        # Added additional dense layers with increased units
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dropout(0.2),  # Additional dropout layer for regularization
        
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dropout(0.1),  # Keep dropout to prevent overfitting
        
        # Additional layer to increase model depth
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.1),  # Keep dropout to prevent overfitting

        tf.keras.layers.Dense(64, activation='relu'),
        
        # Output layer for multi-class classification with softmax activation
        tf.keras.layers.Dense(output_units, activation='softmax')
    ])
    return model

In [ ]:

X, y = load_data('../data/df_aug_shuff_c1.csv')  # Load your dataset

# Split the dataset into training and validation sets
x_train, x_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize the model
num_classes = 5  
model = build_dnn_model(output_units=num_classes)
model.compile(optimizer=Adam(1e-4), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(x_train, y_train, validation_data=(x_val, y_val), batch_size=128, epochs=20, verbose=1)

# Evaluate the model
loss, accuracy = model.evaluate(x_val, y_val, verbose=0)
print(f"Validation loss: {loss}, Validation accuracy: {accuracy}")

predictions = model.predict(x_val)
predicted_classes = np.argmax(predictions, axis=1)
cm = confusion_matrix(y_val, predicted_classes)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title('Confusion Matrix')
plt.show()


In [ ]:
# Plot training & validation accuracy values
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()


In [ ]:
# Plot training & validation loss values
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()


In [ ]:
# Get predictions for all classes using the predict method
predictions = model.predict(x_val)

# Binarize the labels for multiclass
y_bin = label_binarize(y_val, classes=np.unique(y_val))
n_classes = y_bin.shape[1]

# Plot Precision-Recall curve for each class
for i in range(n_classes):
    precision, recall, thresholds = precision_recall_curve(y_bin[:, i], predictions[:, i])
    
    plt.plot(recall, precision, marker='.', label=f'Class {i}')

plt.title('Precision-Recall Curve for Each Class')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.legend(loc='best')
plt.show()


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Train a random forest classifier (as an example)
clf = RandomForestClassifier()
clf.fit(x_train, y_train)

# Plotting feature importance
importances = clf.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure()
plt.title('Feature Importances')
plt.bar(range(x_train.shape[1]), importances[indices],
        color="r", align="center")
plt.xticks(range(x_train.shape[1]), indices)
plt.xlim([-1, x_train.shape[1]])
plt.show()


In [ ]:

from sklearn.metrics import f1_score

f1_scores = 2 * (precision * recall) / (precision + recall)
# Find the index of the maximum F1 score
max_f1_index = np.argmax(f1_scores)
# Get the threshold for the maximum F1 score
best_threshold = thresholds[max_f1_index]

print(f"Best F1 Score: {f1_scores[max_f1_index]} at threshold: {best_threshold}")
# Calculate F1 score for each class
f1_scores_per_class = [f1_score(y_bin[:, i], (predictions[:, i] > threshold).astype(int)) for i in range(n_classes)]

# Calculate macro-averaged F1 score
macro_f1_score = np.mean(f1_scores_per_class)

print(f"Macro-averaged F1 Score: {macro_f1_score}")

f1_scores_per_class = [f1_score(y_bin[:, i], (predictions[:, i] > threshold).astype(int)) for i in range(n_classes)]

# Calculate macro-averaged F1 score
macro_f1_score = np.mean(f1_scores_per_class)

print(f"Macro-averaged F1 Score: {macro_f1_score}")


In [ ]:
# Assuming y_val is your true multiclass labels and predictions are your model's probabilities
# Binarize the labels
y_bin = label_binarize(y_val, classes=np.unique(y_val))

# Assuming predictions have shape [n_samples, n_classes] and contain probabilities
# You will need to convert probabilities to binary predictions
binary_predictions = (predictions > threshold).astype(int)

# Calculate macro-average precision
macro_precision = precision_score(y_bin, binary_predictions, average='macro')

print(f"Macro-average Precision: {macro_precision}")


In [ ]:
# Calculate micro-average precision
micro_precision = precision_score(y_bin, binary_predictions, average='micro')

print(f"Micro-average Precision: {micro_precision}")


In [ ]:
# Calculate weighted-average precision
weighted_precision = precision_score(y_bin, binary_predictions, average='weighted')

print(f"Weighted-average Precision: {weighted_precision}")


In [ ]:
from sklearn.metrics import hamming_loss
from sklearn.metrics import classification_report
ham_loss = hamming_loss(y_bin, binary_predictions)
report = classification_report(y_bin, binary_predictions)
print(report)
print(ham_loss)

In [ ]:
from sklearn.metrics import log_loss
logloss = log_loss(y_bin, binary_predictions)  # y_probs are the predicted probabilities
print(logloss)


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Assuming y_val contains the true labels and predictions contains the predicted labels.
# Both should be 1D arrays with the same length.

# Calculate precision, recall, and F1 score for each class
precision_per_class = precision_score(y_val, predictions, average=None)
recall_per_class = recall_score(y_val, predictions, average=None)
f1_score_per_class = f1_score(y_val, predictions, average=None)

# Macro-average: Calculate metrics for each label, and find their unweighted mean. This does not take label imbalance into account.
precision_macro = precision_score(y_val, predictions, average='macro')
recall_macro = recall_score(y_val, predictions, average='macro')
f1_score_macro = f1_score(y_val, predictions, average='macro')

# Weighted-average: Calculate metrics for each label, and find their average weighted by support (the number of true instances for each label).
precision_weighted = precision_score(y_val, predictions, average='weighted')
recall_weighted = recall_score(y_val, predictions, average='weighted')
f1_score_weighted = f1_score(y_val, predictions, average='weighted')

# Micro-average: Calculate metrics globally by counting the total true positives, false negatives and false positives.
precision_micro = precision_score(y_val, predictions, average='micro')
recall_micro = recall_score(y_val, predictions, average='micro')
f1_score_micro = f1_score(y_val, predictions, average='micro')

print("Per Class Precision:", precision_per_class)
print("Per Class Recall:", recall_per_class)
print("Per Class F1 Score:", f1_score_per_class)
print("Macro-average Precision:", precision_macro)
print("Macro-average Recall:", recall_macro)
print("Macro-average F1 Score:", f1_score_macro)
print("Weighted-average Precision:", precision_weighted)
print("Weighted-average Recall:", recall_weighted)
print("Weighted-average F1 Score:", f1_score_weighted)
print("Micro-average Precision:", precision_micro)
print("Micro-average Recall:", recall_micro)
print("Micro-average F1 Score:", f1_score_micro)


In [ ]:
print(predictions)

In [ ]:
# Assuming binary classification and `predictions` contains output probabilities for the positive class
plt.hist(predictions, bins=10, alpha=0.5, label='Prediction probabilities')
plt.xlabel('Probability of positive class')
plt.ylabel('Frequency')
plt.title('Prediction Probabilities Histogram')
plt.legend()
plt.show()
